In [1]:
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.data import HeteroData
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import RGCNConv
import numpy as np
import random
import pickle
import os
 
SEED = 42
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
 
DATA_DIR = 'Data/esco'   # adjust if your paths differ
ONET_DIR = 'Data/onet'
os.makedirs('outputs', exist_ok=True)

c:\Users\kanis\Desktop\SkillBridge\skillbridge-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu


In [ ]:
DATA_DIR = r'C:\Users\kanis\Desktop\SkillBridge\Data\esco'
ONET_DIR = r'C:\Users\kanis\Desktop\SkillBridge\Data\onet'
skills          = pd.read_csv(f'{DATA_DIR}/skills_en.csv')
occupations     = pd.read_csv(f'{DATA_DIR}/occupations_en.csv')
occ_skill_rels  = pd.read_csv(f'{DATA_DIR}/occupationSkillRelations_en.csv')
skill_rels      = pd.read_csv(f'{DATA_DIR}/skillSkillRelations_en.csv')
onet_occ        = pd.read_csv(f'{ONET_DIR}/Occupation Data.txt', sep='\t')
onet_sw         = pd.read_csv(f'{ONET_DIR}/Software Skills.txt', sep='\t')
 
print(f"ESCO Skills: {len(skills)}")
print(f"ESCO Occupations: {len(occupations)}")
print(f"Skill-Skill relations: {len(skill_rels)}")
print(f"Occ-Skill relations: {len(occ_skill_rels)}")
 
skill_label_map = dict(zip(skills['conceptUri'], skills['preferredLabel']))
occ_label_map   = dict(zip(occupations['conceptUri'], occupations['preferredLabel']))
 

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\kanis\\Desktop\\SkillBridge\\Data\\ONET/Occupation_Data.txt'

In [ ]:
cs_occs     = occupations[occupations['iscoGroup'].astype(str).str.startswith('25')]
cs_occ_uris = set(cs_occs['conceptUri'])
cs_skill_uris = set(
    occ_skill_rels[occ_skill_rels['occupationUri'].isin(cs_occ_uris)]['skillUri']
)
 
G = nx.MultiDiGraph()
for uri in cs_skill_uris:
    G.add_node(uri, node_type='skill', label=skill_label_map.get(uri, uri), source='esco')
for uri in cs_occ_uris:
    G.add_node(uri, node_type='occupation', label=occ_label_map.get(uri, uri), source='esco')
 
for _, row in occ_skill_rels.iterrows():
    if row['occupationUri'] in cs_occ_uris and row['skillUri'] in cs_skill_uris:
        G.add_edge(row['skillUri'], row['occupationUri'], edge_type=row['relationType'])
 
# NOTE: 'optional'/'essential' here are ESCO's skill-relatedness labels,
# used as a broader/narrower proxy per the Week 1 plan -- not true hierarchy.
for _, row in skill_rels.iterrows():
    if row['originalSkillUri'] in cs_skill_uris and row['relatedSkillUri'] in cs_skill_uris:
        G.add_edge(row['originalSkillUri'], row['relatedSkillUri'], edge_type=row['relationType'])
 
onet_cs_codes = set(onet_occ[onet_occ['O*NET-SOC Code'].str.startswith('15-')]['O*NET-SOC Code'])
onet_sw_cs    = onet_sw[onet_sw['O*NET-SOC Code'].isin(onet_cs_codes)]
 
for name in onet_sw_cs['Element Name'].unique():
    G.add_node(f'onet_skill_{name}', node_type='skill', label=name, source='onet')
for code in onet_cs_codes:
    title = onet_occ[onet_occ['O*NET-SOC Code'] == code]['Title'].values
    G.add_node(f'onet_occ_{code}', node_type='occupation',
               label=title[0] if len(title) > 0 else code, source='onet')
for _, row in onet_sw_cs.iterrows():
    G.add_edge(f"onet_skill_{row['Element Name']}", f"onet_occ_{row['O*NET-SOC Code']}",
               edge_type='onet_software', source='onet')
 
G.remove_nodes_from(list(nx.isolates(G)))
 
skill_count = sum(1 for _, d in G.nodes(data=True) if d['node_type'] == 'skill')
occ_count   = sum(1 for _, d in G.nodes(data=True) if d['node_type'] == 'occupation')
print(f"\nGraph: {skill_count} skills, {occ_count} occupations, {G.number_of_edges()} edges")

In [ ]:
all_skill_nodes = [n for n, d in G.nodes(data=True) if d['node_type'] == 'skill']
all_occ_nodes   = [n for n, d in G.nodes(data=True) if d['node_type'] == 'occupation']
skill_idx = {uri: i for i, uri in enumerate(all_skill_nodes)}
occ_idx   = {uri: i for i, uri in enumerate(all_occ_nodes)}
 
data = HeteroData()
data['skill'].x = torch.arange(len(all_skill_nodes)).unsqueeze(1).float()
data['occupation'].x = torch.arange(len(all_occ_nodes)).unsqueeze(1).float()
data['skill'].num_nodes = len(all_skill_nodes)
data['occupation'].num_nodes = len(all_occ_nodes)
 
essential_edges, optional_edges, broader_edges, narrower_edges = [], [], [], []
 
for _, row in occ_skill_rels.iterrows():
    s, o = row['skillUri'], row['occupationUri']
    if s in skill_idx and o in occ_idx:
        (essential_edges if row['relationType'] == 'essential' else optional_edges)\
            .append((skill_idx[s], occ_idx[o]))
 
for _, row in skill_rels.iterrows():
    s1, s2 = row['originalSkillUri'], row['relatedSkillUri']
    if s1 in skill_idx and s2 in skill_idx:
        (broader_edges if row['relationType'] == 'optional' else narrower_edges)\
            .append((skill_idx[s1], skill_idx[s2]))
 
for _, row in onet_sw_cs.iterrows():
    s_id = f"onet_skill_{row['Element Name']}"
    o_id = f"onet_occ_{row['O*NET-SOC Code']}"
    if s_id in skill_idx and o_id in occ_idx:
        essential_edges.append((skill_idx[s_id], occ_idx[o_id]))
 
def to_edge_index(edges):
    if not edges:
        return torch.zeros((2, 0), dtype=torch.long)
    src, dst = zip(*edges)
    return torch.tensor([list(src), list(dst)], dtype=torch.long)
 
data['skill', 'required_for', 'occupation'].edge_index = to_edge_index(essential_edges)
data['skill', 'optional_for', 'occupation'].edge_index = to_edge_index(optional_edges)
data['skill', 'broader', 'skill'].edge_index           = to_edge_index(broader_edges)
data['skill', 'narrower', 'skill'].edge_index          = to_edge_index(narrower_edges)
 
print("\n=== PyG HeteroData Summary ===")
print(data)

In [ ]:
transform = RandomLinkSplit(
    num_val=0.15, num_test=0.15, is_undirected=False,
    edge_types=[
        ('skill', 'required_for', 'occupation'),
        ('skill', 'optional_for', 'occupation'),
        ('skill', 'broader', 'skill'),
        ('skill', 'narrower', 'skill'),
    ],
    rev_edge_types=[None, None, None, None],
)
train_data, val_data, test_data = transform(data)
 
for rel in [('skill', 'required_for', 'occupation'), ('skill', 'optional_for', 'occupation'),
            ('skill', 'broader', 'skill'), ('skill', 'narrower', 'skill')]:
    d_n  = data[rel].edge_index.shape[1]
    tr_n = train_data[rel].edge_index.shape[1]
    v_n  = val_data[rel].edge_index.shape[1]
    te_n = test_data[rel].edge_index.shape[1]
    print(f"{rel[1]:<15} data={d_n:>6} train={tr_n:>6} val={v_n:>6} test={te_n:>6}")
 
with open('outputs/heterodata_rebuilt.pkl', 'wb') as f:
    pickle.dump({
        'data': data, 'train': train_data, 'val': val_data, 'test': test_data,
        'skill_idx': skill_idx, 'occ_idx': occ_idx,
        'all_skill_nodes': all_skill_nodes, 'all_occ_nodes': all_occ_nodes,
        'skill_label_map': skill_label_map, 'occ_label_map': occ_label_map,
    }, f)
print("Saved outputs/heterodata_rebuilt.pkl")

In [ ]:
num_skills   = len(all_skill_nodes)
num_occs     = len(all_occ_nodes)
num_entities = num_skills + num_occs
occ_offset   = num_skills
 
RELATIONS = {'required_for': 0, 'optional_for': 1, 'broader': 2, 'narrower': 3}
REL_NAMES = {v: k for k, v in RELATIONS.items()}
REL_TAIL_TYPE = {0: 'occupation', 1: 'occupation', 2: 'skill', 3: 'skill'}
REL_HEAD_TYPE = {0: 'skill', 1: 'skill', 2: 'skill', 3: 'skill'}
num_relations = len(RELATIONS)
 
def build_triples(pyg_data):
    triples = []
    for rel_name, rel_id in [('required_for', 0), ('optional_for', 1)]:
        edge_key = ('skill', rel_name, 'occupation')
        if edge_key in pyg_data.edge_types:
            ei = pyg_data[edge_key].edge_index
            for i in range(ei.shape[1]):
                triples.append((ei[0, i].item(), rel_id, ei[1, i].item() + occ_offset))
    for rel_name, rel_id in [('broader', 2), ('narrower', 3)]:
        edge_key = ('skill', rel_name, 'skill')
        if edge_key in pyg_data.edge_types:
            ei = pyg_data[edge_key].edge_index
            for i in range(ei.shape[1]):
                triples.append((ei[0, i].item(), rel_id, ei[1, i].item()))
    return triples
 
train_triples = build_triples(train_data)
val_triples   = build_triples(val_data)
test_triples  = build_triples(test_data)
all_known_triples = set(build_triples(data))
 
print(f"\nTrain triples: {len(train_triples)}  Val: {len(val_triples)}  Test: {len(test_triples)}")
 
SKILL_RANGE = torch.arange(0, num_skills)
OCC_RANGE   = torch.arange(occ_offset, occ_offset + num_occs)
def candidates_for(node_type):
    return OCC_RANGE if node_type == 'occupation' else SKILL_RANGE
def _range_for(node_type):
    if node_type == 'occupation':
        return occ_offset, occ_offset + num_occs
    return 0, num_skills
 
# type-constrained negative sampling, used identically for ALL THREE models
def corrupt_triple(h, r, t, true_triples_set):
    while True:
        if random.random() < 0.5:
            lo, hi = _range_for(REL_HEAD_TYPE[r])
            new_h = random.randint(lo, hi - 1)
            if (new_h, r, t) not in true_triples_set:
                return (new_h, r, t)
        else:
            lo, hi = _range_for(REL_TAIL_TYPE[r])
            new_t = random.randint(lo, hi - 1)
            if (h, r, new_t) not in true_triples_set:
                return (h, r, new_t)
 
EMBEDDING_DIM = 100
EPOCHS        = 100
LR            = 0.01
BATCH_SIZE    = 128

In [ ]:
def evaluate(score_fn, test_triples, tag=""):
    per_relation_ranks = {r: [] for r in RELATIONS.values()}
    for h, r, t in test_triples:
        cand_t = candidates_for(REL_TAIL_TYPE[r])
        scores_t = score_fn(torch.full((len(cand_t),), h, dtype=torch.long),
                             torch.full((len(cand_t),), r, dtype=torch.long), cand_t)
        true_idx = (cand_t == t).nonzero(as_tuple=True)[0].item()
        true_score_t = scores_t[true_idx].clone()
        for i, cand in enumerate(cand_t.tolist()):
            if cand != t and (h, r, cand) in all_known_triples:
                scores_t[i] = -1e9
        rank_t = (scores_t > true_score_t).sum().item() + 1
        per_relation_ranks[r].append(rank_t)
 
        cand_h = candidates_for(REL_HEAD_TYPE[r])
        scores_h = score_fn(cand_h, torch.full((len(cand_h),), r, dtype=torch.long),
                             torch.full((len(cand_h),), t, dtype=torch.long))
        true_idx = (cand_h == h).nonzero(as_tuple=True)[0].item()
        true_score_h = scores_h[true_idx].clone()
        for i, cand in enumerate(cand_h.tolist()):
            if cand != h and (cand, r, t) in all_known_triples:
                scores_h[i] = -1e9
        rank_h = (scores_h > true_score_h).sum().item() + 1
        per_relation_ranks[r].append(rank_h)
 
    all_ranks = np.array([r for ranks in per_relation_ranks.values() for r in ranks], dtype=float)
    results = {'model': tag, 'MRR': float(np.mean(1.0 / all_ranks)),
               'Hits@1': float(np.mean(all_ranks <= 1)), 'Hits@3': float(np.mean(all_ranks <= 3)),
               'Hits@10': float(np.mean(all_ranks <= 10))}
    per_rel = []
    for rid, ranks in per_relation_ranks.items():
        if not ranks: continue
        ranks = np.array(ranks, dtype=float)
        per_rel.append({'model': tag, 'relation': REL_NAMES[rid], 'n': len(ranks),
                         'MRR': float(np.mean(1.0 / ranks)), 'Hits@1': float(np.mean(ranks <= 1))})
    return results, per_rel

In [ ]:
class TransE(nn.Module):
    def __init__(self, n_ent, n_rel, dim=100):
        super().__init__()
        self.entity_embeddings   = nn.Embedding(n_ent, dim)
        self.relation_embeddings = nn.Embedding(n_rel, dim)
    def score(self, h, r, t):
        return -torch.norm(self.entity_embeddings(h) + self.relation_embeddings(r)
                            - self.entity_embeddings(t), p=2, dim=1)
 
def train_margin_model(model, name):
    optimizer = optim.Adam(model.parameters(), lr=LR)
    loss_history = []
    print(f"\nTraining {name}...")
    for epoch in range(1, EPOCHS + 1):
        model.train()
        random.shuffle(train_triples)
        epoch_loss, n_batches = 0.0, 0
        for i in range(0, len(train_triples), BATCH_SIZE):
            batch = train_triples[i:i + BATCH_SIZE]
            pos_h = torch.tensor([t[0] for t in batch], dtype=torch.long)
            pos_r = torch.tensor([t[1] for t in batch], dtype=torch.long)
            pos_t = torch.tensor([t[2] for t in batch], dtype=torch.long)
            neg = [corrupt_triple(h, r, t, all_known_triples) for h, r, t in batch]
            neg_h = torch.tensor([x[0] for x in neg], dtype=torch.long)
            neg_r = torch.tensor([x[1] for x in neg], dtype=torch.long)
            neg_t = torch.tensor([x[2] for x in neg], dtype=torch.long)
 
            optimizer.zero_grad()
            pos_score = model.score(pos_h, pos_r, pos_t)
            neg_score = model.score(neg_h, neg_r, neg_t)
            loss = torch.mean(torch.relu(1.0 - pos_score + neg_score))
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item(); n_batches += 1
        loss_history.append(epoch_loss / n_batches)
        if epoch % 20 == 0:
            print(f"  Epoch {epoch:3d}/{EPOCHS} | Loss: {loss_history[-1]:.4f}")
    return loss_history
 
transe = TransE(num_entities, num_relations, EMBEDDING_DIM)
transe_loss = train_margin_model(transe, "TransE")
 
def transe_score_fn(h, r, t):
    with torch.no_grad():
        return transe.score(h, r, t)
transe_results, transe_per_rel = evaluate(transe_score_fn, test_triples, "TransE")

In [ ]:
class DistMult(nn.Module):
    def __init__(self, n_ent, n_rel, dim=100):
        super().__init__()
        self.entity_embeddings   = nn.Embedding(n_ent, dim)
        self.relation_embeddings = nn.Embedding(n_rel, dim)
        nn.init.uniform_(self.entity_embeddings.weight, -6/np.sqrt(dim), 6/np.sqrt(dim))
        nn.init.uniform_(self.relation_embeddings.weight, -6/np.sqrt(dim), 6/np.sqrt(dim))
    def score(self, h, r, t):
        return torch.sum(self.entity_embeddings(h) * self.relation_embeddings(r)
                          * self.entity_embeddings(t), dim=1)
 
distmult = DistMult(num_entities, num_relations, EMBEDDING_DIM)
distmult_loss = train_margin_model(distmult, "DistMult")
 
def distmult_score_fn(h, r, t):
    with torch.no_grad():
        return distmult.score(h, r, t)
distmult_results, distmult_per_rel = evaluate(distmult_score_fn, test_triples, "DistMult")

In [ ]:
class RGCN(nn.Module):
    def __init__(self, n_ent, n_rel, dim):
        super().__init__()
        self.entity_embeddings = nn.Embedding(n_ent, dim)
        self.conv1 = RGCNConv(dim, dim, n_rel)
        self.conv2 = RGCNConv(dim, dim, n_rel)
        self.relation_embeddings = nn.Embedding(n_rel, dim)
        nn.init.xavier_uniform_(self.entity_embeddings.weight)
        nn.init.xavier_uniform_(self.relation_embeddings.weight)
    def encode(self, edge_index, edge_type):
        x = F.relu(self.conv1(self.entity_embeddings.weight, edge_index, edge_type))
        return self.conv2(x, edge_index, edge_type)
    def score(self, h_emb, r_emb, t_emb):
        return -torch.norm(h_emb + r_emb - t_emb, p=2, dim=1)
 
train_h = torch.tensor([t[0] for t in train_triples], dtype=torch.long)
train_t = torch.tensor([t[2] for t in train_triples], dtype=torch.long)
train_r = torch.tensor([t[1] for t in train_triples], dtype=torch.long)
train_edge_index = torch.stack([train_h, train_t], dim=0)
train_edge_type  = train_r
 
rgcn = RGCN(num_entities, num_relations, EMBEDDING_DIM)
rgcn_optimizer = optim.Adam(rgcn.parameters(), lr=LR)
rgcn_loss_history = []
print("\nTraining R-GCN...")
for epoch in range(1, EPOCHS + 1):
    rgcn.train()
    rgcn_optimizer.zero_grad()
    entity_embs = rgcn.encode(train_edge_index, train_edge_type)
    r_embs = rgcn.relation_embeddings.weight
    pos_h, pos_t, pos_r = train_edge_index[0], train_edge_index[1], train_edge_type
    neg = [corrupt_triple(h.item(), r.item(), t.item(), all_known_triples)
           for h, r, t in zip(pos_h, pos_r, pos_t)]
    neg_h = torch.tensor([x[0] for x in neg], dtype=torch.long)
    neg_r = torch.tensor([x[1] for x in neg], dtype=torch.long)
    neg_t = torch.tensor([x[2] for x in neg], dtype=torch.long)
    pos_score = rgcn.score(entity_embs[pos_h], r_embs[pos_r], entity_embs[pos_t])
    neg_score = rgcn.score(entity_embs[neg_h], r_embs[neg_r], entity_embs[neg_t])
    loss = torch.mean(torch.relu(1.0 - pos_score + neg_score))
    loss.backward()
    rgcn_optimizer.step()
    rgcn_loss_history.append(loss.item())
    if epoch % 20 == 0:
        print(f"  Epoch {epoch:3d}/{EPOCHS} | Loss: {loss.item():.4f}")
 
with torch.no_grad():
    rgcn_entity_embs = rgcn.encode(train_edge_index, train_edge_type)
    rgcn_r_embs = rgcn.relation_embeddings.weight
def rgcn_score_fn(h, r, t):
    with torch.no_grad():
        return rgcn.score(rgcn_entity_embs[h], rgcn_r_embs[r], rgcn_entity_embs[t])
rgcn_results, rgcn_per_rel = evaluate(rgcn_score_fn, test_triples, "R-GCN")

In [ ]:
comparison = pd.DataFrame([transe_results, distmult_results, rgcn_results])
print("\n=== FINAL 3-Model Comparison (type-constrained, filtered, identical protocol) ===")
print(comparison.to_string(index=False))
comparison.to_csv('outputs/final_comparison.csv', index=False)
 
per_rel_all = pd.DataFrame(transe_per_rel + distmult_per_rel + rgcn_per_rel)
print("\n=== Per-Relation Hits@1 ===")
print(per_rel_all.pivot(index='relation', columns='model', values='Hits@1').to_string())
per_rel_all.to_csv('outputs/final_per_relation.csv', index=False)
 
torch.save({'model_state_dict': transe.state_dict(), 'num_entities': num_entities,
            'num_relations': num_relations, 'embedding_dim': EMBEDDING_DIM},
           'outputs/transe_final.pt')
torch.save({'model_state_dict': distmult.state_dict(), 'num_entities': num_entities,
            'num_relations': num_relations, 'embedding_dim': EMBEDDING_DIM},
           'outputs/distmult_final.pt')
torch.save({'model_state_dict': rgcn.state_dict(), 'num_entities': num_entities,
            'num_relations': num_relations, 'embedding_dim': EMBEDDING_DIM,
            'train_edge_index': train_edge_index, 'train_edge_type': train_edge_type},
           'outputs/rgcn_final.pt')
 
print("\nAll checkpoints and results saved to outputs/. Rebuild complete.")